# 05 — External Dataset Bonus: Shark Attacks vs. International Tourism Arrivals

- **Owner:** Romain Thepaut (`@RomThpt`)
- **Course Rubric:** Task 6 — External Dataset `[+1 extra point]`
- **External Source:** World Bank Indicator `ST.INT.ARVL` — *International tourism, number of arrivals* (1995-2020, 266 countries/regions)

In [1]:
# Import libraries and project functions
import os
import sys
import warnings
import pandas as pd
import plotly.express as px

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_shark_data, load_tourism_data, clean_shark_data, clean_and_merge_tourism
from src.indicators import query_tourism_enrichment

warnings.filterwarnings('ignore')
clean_sharks = clean_shark_data(load_shark_data('../data/attacks.csv'))
raw_tourism = load_tourism_data('../data/tourism_arrivals.csv')
merged_tourism = clean_and_merge_tourism(clean_sharks, raw_tourism)
print(f'Merged dataset ready: {len(merged_tourism)} country-year observations ({merged_tourism["Year"].min()}-{merged_tourism["Year"].max()}).')

Merged dataset ready: 1365 country-year observations (1995-2017).


## External bonus — Reported incidents per million arrivals

Use complete years from 1995 to 2017 for countries and territories present in both sources. Keep years with positive recorded arrivals, including years with zero incident reports. Sum reports and arrivals by geography, divide, and scale by one million.

Pearson and Spearman correlations describe the pooled country-year panel. Incident reports cover residents and visitors. The aggregate ratio uses arrival counts as a measure of tourism volume.

In [2]:
# Execute External Tourism Bonus analysis
bonus = query_tourism_enrichment(merged_tourism)

print(f"Global Pearson correlation (Arrivals vs. Attacks): r = {bonus['pearson_corr']:.4f} (p = {bonus['pearson_pval']:.4e})")
print(f"Global Spearman rank correlation: rho = {bonus['spearman_corr']:.4f} (p = {bonus['spearman_pval']:.4e})")

print('\n--- Top 8 by volume — Reported incidents per million arrivals ---')
display(bonus['country_comparison'])
bonus['fig_comparison'].show()
bonus['fig_dual_axis'].show()

Global Pearson correlation (Arrivals vs. Attacks): r = 0.3841 (p = 3.1544e-49)
Global Spearman rank correlation: rho = 0.1645 (p = 9.6763e-10)

--- Top 8 by volume — Reported incidents per million arrivals ---


,Country_clean,Years_Observed,Total_Attacks,Total_Tourist_Arrivals,attacks_per_1M_tourists
0,United States,23,1170,2.870403e+09,0.408
1,Australia,23,410,1.287050e+08,3.186
2,South Africa,23,169,2.127390e+08,0.794
3,Brazil,23,73,1.134390e+08,0.644
4,"Bahamas, The",23,57,1.104050e+08,0.516
5,New Zealand,23,51,5.352300e+07,0.953
6,New Caledonia,23,29,2.399000e+06,12.088
7,Mexico,23,27,2.112290e+09,0.013


In [3]:
# Additional cross-country view: annual evolution of reports per million arrivals for Australia, South Africa, and US
focus_countries = ['Australia', 'South Africa', 'United States', 'New Zealand']
trend_df = bonus['enriched_df'][bonus['enriched_df']['Country_clean'].isin(focus_countries)].copy()
fig_rate_trend = px.line(
    trend_df,
    x='Year',
    y='attacks_per_1M_tourists',
    color='Country_clean',
    markers=True,
    labels={'attacks_per_1M_tourists': 'Reports per Million Arrivals', 'Country_clean': 'Country'},
    title='Annual reported incidents per million arrivals (1995–2017)',
)
fig_rate_trend.update_layout(template='plotly_white')
fig_rate_trend.show()